In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel
from torch.utils.data import DataLoader,TensorDataset

device = "cpu"

config = {
          "embd":256,
          "con_length":128,
          "qkv_bias":False,
          "heads":8,
          "head_dim":32,
          "vocab_size":20000,
          "drop_rate":0.1,
          "batch_size":4,
          "layers":4
          }

with open("C:/Users/block/Desktop/AI Universe/AI Datasets/dataset.txt","r",encoding="utf-8") as f:
    text = f.read()

tokenizer = Tokenizer.from_file("Tokenizer_data.json")
text_ids = tokenizer.encode(text).ids
data = torch.tensor(text_ids)
split_rate = 0.9
train_data = data[:int(len(data) * 0.9)]
val_data = data[len(train_data):len(data)]




def get_batch(train = True):
    data = train_data if train == True else val_data
    ix = torch.randint(0,len(data)-config["con_length"]-1,(config["batch_size"],))
    x = torch.stack([data[i:i+config["con_length"]] for i in ix])
    y = torch.stack([data[i+1:i+config["con_length"]+1] for i in ix])

    return x.to(device),y.to(device)


class Attention(nn.Module):
    def __init__(self,d_in = config["embd"],d_out=config["embd"],con_length = config["con_length"],qkv_bias = config["qkv_bias"],drop_rate = config["drop_rate"],heads = config["heads"],head_dim = config["head_dim"]):
        super().__init__()
        self.W_query = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_key = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.heads = heads
        self.head_dim = head_dim
        self.out = nn.Linear(d_out,d_out)
        self.dropout = nn.Dropout(drop_rate)
        self.register_buffer("mask",torch.triu(torch.ones(con_length,con_length),diagonal=1))

    def forward(self,x):
        batch,num_tok,embd = x.shape

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        queries = queries.view(batch,num_tok,self.heads,self.head_dim)
        keys = keys.view(batch,num_tok,self.heads,self.head_dim)
        values = values.view(batch,num_tok,self.heads,self.head_dim)

        queries = queries.transpose(1,2)
        keys = keys.transpose(1,2)
        values = values.transpose(1,2)

        attn_scores = queries @ keys.transpose(2,3)
        masked = self.mask.bool()[:num_tok,:num_tok]
        attn_scores.masked_fill_(masked,-torch.inf)

        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
        attn_weights = (attn_weights @ values).transpose(1,2)
        attn_weights = self.dropout(attn_weights)

        cont_vec = attn_weights.contiguous().view(batch,num_tok,embd)
        cont_vec = self.out(cont_vec)

        return cont_vec
    


class LayerNorm(nn.Module):
    def __init__(self,embd = config["embd"]):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(embd))
        self.shift = nn.Parameter(torch.zeros(embd))

    def forward(self,x):
        mean = x.mean(keepdim = True,dim = -1)
        var = x.var(keepdim = True,dim = -1)
        norm_x = (x-mean)/torch.sqrt(var+self.eps)
        return self.scale * norm_x + self.shift
    

class FeedForward(nn.Module):
    def __init__(self,embd = config["embd"]):
        super().__init__()

        self.f_layer = nn.Sequential(nn.Linear(embd,embd*4),nn.GELU(),nn.Linear(embd*4,embd))

    def forward(self,x):
        return self.f_layer(x)
    


class Transformer(nn.Module):
    def __init__(self,drop_rate = config["drop_rate"]):
        super().__init__()

        self.drop_shortcut = nn.Dropout(drop_rate)
        self.attention = Attention()
        self.layernorm1 = LayerNorm()
        self.layernorm2 = LayerNorm()
        self.ff = FeedForward()


    def forward(self,x):

        shortcut = x
        x = self.layernorm1(x)
        x = self.attention(x)
        x = self.drop_shortcut(x)
        x = shortcut + x

        shortcut = x
        x = self.layernorm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = shortcut + x

        return x




class Model(nn.Module):
    def __init__(self,layers = config["layers"],drop_rate = config["drop_rate"],vocab = config["vocab_size"],embd = config["embd"],con_len = config["con_length"]):
        super().__init__()

        self.drop_emb = nn.Dropout(drop_rate)
        self.out_head = nn.Linear(embd,vocab)
        self.tf = nn.Sequential(*[Transformer() for _ in range(layers)])
        self.tok_emb = nn.Embedding(vocab,embd)
        self.pos_embd = nn.Embedding(con_len,embd)
        self.final_norm = LayerNorm()

    def forward(self,x):
        batch,num_tok = x.shape
        tok_emb = self.tok_emb(x)
        pos_emb = self.pos_embd(torch.arange(num_tok).to(device))
        x = tok_emb + pos_emb
        x = self.drop_emb(x)
        x = self.tf(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits
    

print("Reached Training Loop")

best_val_loss = float("inf")
model = Model().to(device)
optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4)

steps = 20000

for step in range(steps):

    

    get_batch(True)
    x2,y2 = get_batch()
    logits = model(x2)
    train_loss = F.cross_entropy(logits.view(-1,config["vocab_size"]),y2.view(-1))
    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()
    
    with torch.no_grad():
        x1,y1 = get_batch(False)
        logits = model(x1)
        val_loss = F.cross_entropy(logits.view(-1,config["vocab_size"]),y1.view(-1))
    
    

    if step % 50 == 0:
        if val_loss < best_val_loss:
            checkpoint = {
                "model_weights":model.state_dict(),
                "optimizer":optimizer.state_dict()
            }

            torch.save(checkpoint,"Checkpoint.pth")
        print(f"Step:{step}      Train_Loss:{train_loss:.4f}  Val_Loss:{val_loss:.4f}")





   
   

        





